# 09 RAG Evaluation

## Goal

This notebook evaluates the RiskRadar AI RAG system.

The workflow is:

```text
RAG demo answers
→ citation validation
→ source marker validation
→ evidence lookup
→ groundedness proxy scoring
→ retrieval evaluation summary
→ failure case checklist
→ final evaluation report
```

This notebook does not try to prove the system is perfect.

The goal is to show that the system has a responsible evaluation layer:

```text
retrieval quality
+ citation usage
+ answer-evidence alignment
+ known limitations
+ failure cases
```

This evaluation layer shows that the system is designed for grounded, testable, and auditable RAG behavior.

In [1]:
# Import tools for file and folder paths
from pathlib import Path

# Import pandas for working with tables
import pandas as pd

# Import numpy for numerical calculations
import numpy as np

# Import regular expressions for text checks
import re

# Import ast to safely parse citation lists saved as strings
import ast

# Import textwrap for readable text previews
import textwrap

In [2]:
# Detect the project root automatically
# If this notebook is inside the notebooks folder, move one level up
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Create main project paths
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

# Create reports folder if it does not exist
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Print paths to verify everything is correct
print("Project root:", PROJECT_ROOT)
print("Processed data folder:", PROCESSED_DIR)
print("Reports folder:", REPORTS_DIR)

Project root: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI
Processed data folder: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\processed
Reports folder: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\reports


#### Load RAG demo answers

In [3]:
# Set path to demo answers from notebook 08
demo_answers_file = PROCESSED_DIR / "sec_10k_rag_demo_answers.csv"

# Check that the file exists
if not demo_answers_file.exists():
    raise FileNotFoundError(
        f"Could not find {demo_answers_file}. Run 08_rag_answer_generation.ipynb first."
    )

# Load demo answers
demo_answers_df = pd.read_csv(demo_answers_file)

# Preview demo answers
demo_answers_df.head()

,question,ticker,section_name,answer,citations,total_time_seconds
0,What AI and competition risks does NVIDIA ment...,NVDA,item_1a_risk_factors,"Based on the provided SEC evidence, NVIDIA men...","['NVDA 2026-02-25 10-K, item_1a_risk_factors, ...",74.21
1,What cybersecurity risks does Microsoft mention?,MSFT,item_1a_risk_factors,Microsoft mentions the following cybersecurity...,"['MSFT 2025-07-30 10-K, item_1a_risk_factors, ...",94.54
2,What supply chain risks does Tesla mention?,TSLA,item_1a_risk_factors,"Based on the provided SEC evidence, Tesla ment...","['TSLA 2026-01-29 10-K, item_1a_risk_factors, ...",126.40
3,What competition risks does Apple describe?,AAPL,item_1a_risk_factors,"Based on the provided SEC evidence, Apple desc...","['AAPL 2025-10-31 10-K, item_1a_risk_factors, ...",99.38
4,What semiconductor competition risks does AMD ...,AMD,item_1a_risk_factors,"Based on the provided SEC evidence, AMD mentio...","['AMD 2026-02-04 10-K, item_1a_risk_factors, c...",106.97


#### Load final RAG chunks

In [4]:
# Set path to final RAG chunks from notebook 04
rag_chunks_file = PROCESSED_DIR / "sec_10k_rag_chunks.csv"

# Check that the chunk file exists
if not rag_chunks_file.exists():
    raise FileNotFoundError(
        f"Could not find {rag_chunks_file}. Run 04_chunking_experiments.ipynb first."
    )

# Load final chunks
rag_chunks_df = pd.read_csv(rag_chunks_file)

# Fill missing values
rag_chunks_df = rag_chunks_df.fillna("")

# Preview citation metadata
rag_chunks_df[
    [
        "ticker",
        "section_name",
        "citation_label",
        "filing_url",
        "chunk_text"
    ]
].head()

,ticker,section_name,citation_label,filing_url,chunk_text
0,AAPL,item_1_business,"AAPL 2025-10-31 10-K, item_1_business, chunk 0",https://www.sec.gov/Archives/edgar/data/320193...,Item 1. Business Company Background The Compan...
1,AAPL,item_1_business,"AAPL 2025-10-31 10-K, item_1_business, chunk 1",https://www.sec.gov/Archives/edgar/data/320193...,Apple Inc. | 2025 Form 10-K | 1 Services Adver...
2,AAPL,item_1_business,"AAPL 2025-10-31 10-K, item_1_business, chunk 2",https://www.sec.gov/Archives/edgar/data/320193...,"Greater China includes China mainland, Hong Ko..."
3,AAPL,item_1_business,"AAPL 2025-10-31 10-K, item_1_business, chunk 3",https://www.sec.gov/Archives/edgar/data/320193...,by imitating the Company’s products and infrin...
4,AAPL,item_1_business,"AAPL 2025-10-31 10-K, item_1_business, chunk 4",https://www.sec.gov/Archives/edgar/data/320193...,provide products and services at little or no ...


Load answer validation from notebook 08

In [5]:
# Set path to answer validation file from notebook 08
answer_validation_file = PROCESSED_DIR / "sec_10k_rag_answer_validation.csv"

# Load answer validation if it exists
if answer_validation_file.exists():
    answer_validation_df = pd.read_csv(answer_validation_file)
else:
    answer_validation_df = pd.DataFrame()

# Display validation file if available
answer_validation_df.head()

,question,ticker,section_name,answer,citations,total_time_seconds,has_source_marker,num_retrieved_citations
0,What AI and competition risks does NVIDIA ment...,NVDA,item_1a_risk_factors,"Based on the provided SEC evidence, NVIDIA men...","['NVDA 2026-02-25 10-K, item_1a_risk_factors, ...",74.21,True,3
1,What cybersecurity risks does Microsoft mention?,MSFT,item_1a_risk_factors,Microsoft mentions the following cybersecurity...,"['MSFT 2025-07-30 10-K, item_1a_risk_factors, ...",94.54,True,3
2,What supply chain risks does Tesla mention?,TSLA,item_1a_risk_factors,"Based on the provided SEC evidence, Tesla ment...","['TSLA 2026-01-29 10-K, item_1a_risk_factors, ...",126.40,True,3
3,What competition risks does Apple describe?,AAPL,item_1a_risk_factors,"Based on the provided SEC evidence, Apple desc...","['AAPL 2025-10-31 10-K, item_1a_risk_factors, ...",99.38,True,3
4,What semiconductor competition risks does AMD ...,AMD,item_1a_risk_factors,"Based on the provided SEC evidence, AMD mentio...","['AMD 2026-02-04 10-K, item_1a_risk_factors, c...",106.97,True,3


#### Load retrieval evaluations from notebook 06 if available

In [6]:
# Set paths to retrieval evaluation outputs from notebook 06
filtered_eval_file = PROCESSED_DIR / "sec_10k_filtered_retrieval_eval.csv"
unfiltered_eval_file = PROCESSED_DIR / "sec_10k_unfiltered_retrieval_eval.csv"
retrieval_summary_file = PROCESSED_DIR / "sec_10k_retrieval_eval_summary.csv"

# Load filtered retrieval evaluation if it exists
filtered_eval_df = (
    pd.read_csv(filtered_eval_file)
    if filtered_eval_file.exists()
    else pd.DataFrame()
)

# Load unfiltered retrieval evaluation if it exists
unfiltered_eval_df = (
    pd.read_csv(unfiltered_eval_file)
    if unfiltered_eval_file.exists()
    else pd.DataFrame()
)

# Load retrieval summary if it exists
retrieval_summary_df = (
    pd.read_csv(retrieval_summary_file)
    if retrieval_summary_file.exists()
    else pd.DataFrame()
)

# Display retrieval summary
retrieval_summary_df

,method,metric,value
0,filtered,test_questions,8.000000
1,filtered,ticker_hit_rate_at_5,1.000000
2,filtered,section_hit_rate_at_5,1.000000
3,filtered,pair_hit_rate_at_5,1.000000
4,filtered,average_top_distance,0.913938
5,unfiltered,test_questions,8.000000
6,unfiltered,ticker_hit_rate_at_5,1.000000
7,unfiltered,section_hit_rate_at_5,0.875000
8,unfiltered,pair_hit_rate_at_5,0.750000
9,unfiltered,average_top_distance,0.804657


#### Evaluation metrics

## Evaluation Metrics

This notebook uses practical first-version RAG evaluation metrics.

We will check:

```text
1. Answer generation count
2. Citation marker presence
3. Number of retrieved citations
4. Whether citations map back to known chunks
5. Answer-evidence keyword overlap
6. Retrieval evaluation metrics from notebook 06
7. Known failure cases
```

The groundedness score here is a simple proxy.

It does not replace human review, but it helps identify answers that may need inspection.

#### Helper function to parse citation lists

In [7]:
def parse_citations(citation_value):
    """
    Convert saved citation values into a Python list.

    In CSV files, Python lists may be saved as strings.
    Example:
    "['AAPL 2025 10-K...', 'AAPL 2025 10-K...']"
    """

    # Return empty list for missing values
    if pd.isna(citation_value):
        return []

    # If value is already a list, return it
    if isinstance(citation_value, list):
        return citation_value

    # Convert value to string
    citation_text = str(citation_value)

    # Try to parse a Python-style list string
    try:
        parsed_value = ast.literal_eval(citation_text)

        # Return parsed list if successful
        if isinstance(parsed_value, list):
            return parsed_value

    # If parsing fails, continue to fallback
    except Exception:
        pass

    # Fallback: return the value as a one-item list if it is not empty
    if citation_text.strip():
        return [citation_text.strip()]

    # Otherwise return empty list
    return []

#### Helper function for source markers

In [8]:
def extract_source_markers(answer_text):
    """
    Extract source markers from an answer.

    Example markers:
    [Source 1]
    [Source 2]
    """

    # Convert answer to string
    answer_text = str(answer_text)

    # Find source markers
    markers = re.findall(r"\[Source\s+\d+\]", answer_text)

    # Return markers
    return markers

Build citation validation table

In [9]:
# Create a copy of demo answers for evaluation
citation_eval_df = demo_answers_df.copy()

# Parse citation lists
citation_eval_df["parsed_citations"] = citation_eval_df["citations"].apply(parse_citations)

# Count retrieved citations
citation_eval_df["num_citations"] = citation_eval_df["parsed_citations"].apply(len)

# Extract source markers from generated answer
citation_eval_df["source_markers"] = citation_eval_df["answer"].apply(extract_source_markers)

# Count source markers
citation_eval_df["num_source_markers"] = citation_eval_df["source_markers"].apply(len)

# Check whether answer has at least one source marker
citation_eval_df["has_source_marker"] = citation_eval_df["num_source_markers"] > 0

# Check whether answer has at least one retrieved citation
citation_eval_df["has_retrieved_citation"] = citation_eval_df["num_citations"] > 0

# Display citation evaluation
citation_eval_df[
    [
        "question",
        "ticker",
        "num_citations",
        "num_source_markers",
        "has_source_marker",
        "has_retrieved_citation"
    ]
]

,question,ticker,num_citations,num_source_markers,has_source_marker,has_retrieved_citation
0,What AI and competition risks does NVIDIA ment...,NVDA,3,6,True,True
1,What cybersecurity risks does Microsoft mention?,MSFT,3,4,True,True
2,What supply chain risks does Tesla mention?,TSLA,3,3,True,True
3,What competition risks does Apple describe?,AAPL,3,6,True,True
4,What semiconductor competition risks does AMD ...,AMD,3,3,True,True


Check whether citations map to known chunks

In [10]:
# Create a set of valid citation labels from the chunk dataset
valid_citation_labels = set(rag_chunks_df["citation_label"].unique())

def count_valid_citations(citations):
    """
    Count how many retrieved citations exist in the RAG chunk dataset.
    """

    # Count citations that map to known chunk citation labels
    valid_count = sum(
        citation in valid_citation_labels
        for citation in citations
    )

    # Return valid citation count
    return valid_count


# Count valid citations
citation_eval_df["valid_citation_count"] = citation_eval_df["parsed_citations"].apply(
    count_valid_citations
)

# Calculate citation validity rate per answer
citation_eval_df["citation_validity_rate"] = citation_eval_df.apply(
    lambda row: (
        row["valid_citation_count"] / row["num_citations"]
        if row["num_citations"] > 0 else 0
    ),
    axis=1
)

# Display citation validity
citation_eval_df[
    [
        "question",
        "ticker",
        "num_citations",
        "valid_citation_count",
        "citation_validity_rate"
    ]
]

,question,ticker,num_citations,valid_citation_count,citation_validity_rate
0,What AI and competition risks does NVIDIA ment...,NVDA,3,3,1.0
1,What cybersecurity risks does Microsoft mention?,MSFT,3,3,1.0
2,What supply chain risks does Tesla mention?,TSLA,3,3,1.0
3,What competition risks does Apple describe?,AAPL,3,3,1.0
4,What semiconductor competition risks does AMD ...,AMD,3,3,1.0


Text normalization and tokenization

## Groundedness Proxy

A full faithfulness evaluation normally needs human review or an LLM judge.

For this project, we will create a simple proxy score:

```text
answer keywords
compared with
retrieved evidence keywords
```

If the answer uses many terms that appear in the retrieved evidence, that is a positive signal.

This does not prove perfect faithfulness, but it helps flag weak answers.

In [11]:
def normalize_for_eval(text):
    """
    Normalize text for lightweight evaluation.
    """

    # Convert text to lowercase string
    text = str(text).lower()

    # Keep only letters, numbers, and spaces
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Collapse repeated whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # Return normalized text
    return text


def tokenize_for_eval(text):
    """
    Tokenize text for evaluation and remove simple stopwords.
    """

    # Normalize text
    text = normalize_for_eval(text)

    # Split into tokens
    tokens = text.split()

    # Define simple stopwords
    stopwords = {
        "the", "a", "an", "and", "or", "of", "to", "in", "for", "on",
        "with", "as", "by", "from", "that", "this", "it", "is", "are",
        "was", "were", "be", "been", "being", "at", "into", "their",
        "its", "our", "we", "they", "you", "can", "could", "may",
        "might", "will", "would", "should", "source", "evidence"
    }

    # Remove stopwords and very short tokens
    tokens = [
        token for token in tokens
        if token not in stopwords and len(token) > 2
    ]

    # Return tokens
    return tokens

Get evidence text for citations

In [12]:
def get_evidence_text_for_citations(citations):
    """
    Combine chunk text for retrieved citations.
    """

    # Return empty string if no citations exist
    if len(citations) == 0:
        return ""

    # Filter chunks matching citation labels
    evidence_rows = rag_chunks_df[
        rag_chunks_df["citation_label"].isin(citations)
    ]

    # Combine evidence text
    evidence_text = " ".join(evidence_rows["chunk_text"].astype(str).tolist())

    # Return combined evidence text
    return evidence_text

Calculate answer-evidence overlap

In [13]:
def calculate_answer_evidence_overlap(answer_text, evidence_text):
    """
    Calculate simple keyword overlap between an answer and its retrieved evidence.
    """

    # Tokenize answer and evidence
    answer_tokens = set(tokenize_for_eval(answer_text))
    evidence_tokens = set(tokenize_for_eval(evidence_text))

    # Return zero scores if answer has no tokens
    if len(answer_tokens) == 0:
        return {
            "answer_token_count": 0,
            "evidence_token_count": len(evidence_tokens),
            "overlap_token_count": 0,
            "answer_evidence_overlap_rate": 0
        }

    # Calculate overlap
    overlap_tokens = answer_tokens.intersection(evidence_tokens)

    # Calculate overlap rate
    overlap_rate = len(overlap_tokens) / len(answer_tokens)

    # Return overlap metrics
    return {
        "answer_token_count": len(answer_tokens),
        "evidence_token_count": len(evidence_tokens),
        "overlap_token_count": len(overlap_tokens),
        "answer_evidence_overlap_rate": overlap_rate
    }

Build groundedness proxy table

In [14]:
# Create empty list for groundedness records
groundedness_records = []

# Loop through each answer
for _, row in citation_eval_df.iterrows():

    # Get citations for this answer
    citations = row["parsed_citations"]

    # Get combined evidence text
    evidence_text = get_evidence_text_for_citations(citations)

    # Calculate overlap metrics
    overlap_metrics = calculate_answer_evidence_overlap(
        answer_text=row["answer"],
        evidence_text=evidence_text
    )

    # Store record
    groundedness_records.append({
        "question": row["question"],
        "ticker": row["ticker"],
        "section_name": row["section_name"],
        "num_citations": row["num_citations"],
        "valid_citation_count": row["valid_citation_count"],
        "has_source_marker": row["has_source_marker"],
        **overlap_metrics
    })

# Convert records to DataFrame
groundedness_eval_df = pd.DataFrame(groundedness_records)

# Display groundedness proxy evaluation
groundedness_eval_df

,question,ticker,section_name,num_citations,valid_citation_count,has_source_marker,answer_token_count,evidence_token_count,overlap_token_count,answer_evidence_overlap_rate
0,What AI and competition risks does NVIDIA ment...,NVDA,item_1a_risk_factors,3,3,True,51,354,35,0.686275
1,What cybersecurity risks does Microsoft mention?,MSFT,item_1a_risk_factors,3,3,True,50,359,40,0.800000
2,What supply chain risks does Tesla mention?,TSLA,item_1a_risk_factors,3,3,True,68,373,57,0.838235
3,What competition risks does Apple describe?,AAPL,item_1a_risk_factors,3,3,True,58,352,40,0.689655
4,What semiconductor competition risks does AMD ...,AMD,item_1a_risk_factors,3,3,True,42,403,35,0.833333


Add quality flags

In [15]:
# Create a copy for quality flags
quality_flags_df = groundedness_eval_df.copy()

# Flag answers without source markers
quality_flags_df["flag_missing_source_marker"] = ~quality_flags_df["has_source_marker"]

# Flag answers with no valid citations
quality_flags_df["flag_no_valid_citations"] = quality_flags_df["valid_citation_count"] == 0

# Flag answers with low answer-evidence overlap
quality_flags_df["flag_low_overlap"] = quality_flags_df["answer_evidence_overlap_rate"] < 0.35

# Count total flags per answer
quality_flags_df["total_flags"] = (
    quality_flags_df["flag_missing_source_marker"].astype(int)
    + quality_flags_df["flag_no_valid_citations"].astype(int)
    + quality_flags_df["flag_low_overlap"].astype(int)
)

# Display quality flags
quality_flags_df[
    [
        "question",
        "ticker",
        "has_source_marker",
        "valid_citation_count",
        "answer_evidence_overlap_rate",
        "flag_missing_source_marker",
        "flag_no_valid_citations",
        "flag_low_overlap",
        "total_flags"
    ]
]

,question,ticker,has_source_marker,valid_citation_count,answer_evidence_overlap_rate,flag_missing_source_marker,flag_no_valid_citations,flag_low_overlap,total_flags
0,What AI and competition risks does NVIDIA ment...,NVDA,True,3,0.686275,False,False,False,0
1,What cybersecurity risks does Microsoft mention?,MSFT,True,3,0.800000,False,False,False,0
2,What supply chain risks does Tesla mention?,TSLA,True,3,0.838235,False,False,False,0
3,What competition risks does Apple describe?,AAPL,True,3,0.689655,False,False,False,0
4,What semiconductor competition risks does AMD ...,AMD,True,3,0.833333,False,False,False,0


Create evaluation summary

In [16]:
# Create high-level evaluation summary
evaluation_summary = pd.DataFrame({
    "metric": [
        "demo_answers",
        "answers_with_source_markers",
        "source_marker_rate",
        "average_citations_per_answer",
        "average_citation_validity_rate",
        "average_answer_evidence_overlap",
        "answers_with_quality_flags"
    ],
    "value": [
        len(citation_eval_df),
        citation_eval_df["has_source_marker"].sum(),
        citation_eval_df["has_source_marker"].mean(),
        citation_eval_df["num_citations"].mean(),
        citation_eval_df["citation_validity_rate"].mean(),
        groundedness_eval_df["answer_evidence_overlap_rate"].mean(),
        (quality_flags_df["total_flags"] > 0).sum()
    ]
})

# Display evaluation summary
evaluation_summary

,metric,value
0,demo_answers,5.0000
1,answers_with_source_markers,5.0000
2,source_marker_rate,1.0000
3,average_citations_per_answer,3.0000
4,average_citation_validity_rate,1.0000
5,average_answer_evidence_overlap,0.7695
6,answers_with_quality_flags,0.0000


Add retrieval summary if available

In [18]:
# Create a readable retrieval summary table
if not retrieval_summary_df.empty:
    retrieval_eval_summary = retrieval_summary_df.copy()
else:
    retrieval_eval_summary = pd.DataFrame({
        "method": ["not_available"],
        "metric": ["retrieval_summary"],
        "value": ["Run 06_baseline_retrieval.ipynb to generate retrieval evaluation."]
    })

# Display retrieval evaluation summary
retrieval_eval_summary

,method,metric,value
0,filtered,test_questions,8.000000
1,filtered,ticker_hit_rate_at_5,1.000000
2,filtered,section_hit_rate_at_5,1.000000
3,filtered,pair_hit_rate_at_5,1.000000
4,filtered,average_top_distance,0.913938
5,unfiltered,test_questions,8.000000
6,unfiltered,ticker_hit_rate_at_5,1.000000
7,unfiltered,section_hit_rate_at_5,0.875000
8,unfiltered,pair_hit_rate_at_5,0.750000
9,unfiltered,average_top_distance,0.804657


## Failure Case Checklist

A serious RAG project should explain where the system can fail.

Known possible failure cases:

```text
1. Retrieved evidence may be relevant but incomplete.
2. The local LLM may summarize too broadly.
3. Citations may be present even if the wording needs human review.
4. Keyword overlap is only a proxy, not proof of faithfulness.
5. SEC filings are long and section extraction may not be perfect.
6. The system currently covers only the companies downloaded into the vector store.
7. The system answers from filings only, not live market news.
```

This is honest and professional.

This evaluation layer shows that the system is designed for grounded, testable, and auditable RAG behavior.

Create failure case table

In [19]:
# Create failure case review table
failure_cases_df = pd.DataFrame([
    {
        "failure_case": "Question asks about a company not in the vector database",
        "example": "What does Amazon say about competition?",
        "expected_behavior": "System should say no evidence is available unless Amazon filings were ingested.",
        "mitigation": "Validate ticker coverage before retrieval."
    },
    {
        "failure_case": "Question asks for current news",
        "example": "What happened to NVIDIA stock today?",
        "expected_behavior": "System should say SEC filings do not answer live market questions.",
        "mitigation": "Add a separate live-data or news retrieval layer."
    },
    {
        "failure_case": "Evidence is retrieved from the right company but wrong section",
        "example": "Risk question retrieves MD&A instead of Risk Factors.",
        "expected_behavior": "Use section filters when the user intent is clear.",
        "mitigation": "Add query routing to choose likely SEC sections."
    },
    {
        "failure_case": "LLM makes a claim not clearly supported by evidence",
        "example": "Answer adds business interpretation beyond the chunks.",
        "expected_behavior": "Flag for human review or stricter prompting.",
        "mitigation": "Use faithfulness evaluation and quote-level citation checks."
    },
    {
        "failure_case": "Chunk boundary cuts off context",
        "example": "Important sentence starts in one chunk and ends in another.",
        "expected_behavior": "Overlap should reduce this issue.",
        "mitigation": "Improve with sentence-aware or semantic chunking."
    }
])

# Display failure cases
failure_cases_df

,failure_case,example,expected_behavior,mitigation
0,Question asks about a company not in the vecto...,What does Amazon say about competition?,System should say no evidence is available unl...,Validate ticker coverage before retrieval.
1,Question asks for current news,What happened to NVIDIA stock today?,System should say SEC filings do not answer li...,Add a separate live-data or news retrieval layer.
2,Evidence is retrieved from the right company b...,Risk question retrieves MD&A instead of Risk F...,Use section filters when the user intent is cl...,Add query routing to choose likely SEC sections.
3,LLM makes a claim not clearly supported by evi...,Answer adds business interpretation beyond the...,Flag for human review or stricter prompting.,Use faithfulness evaluation and quote-level ci...
4,Chunk boundary cuts off context,Important sentence starts in one chunk and end...,Overlap should reduce this issue.,Improve with sentence-aware or semantic chunking.


## Manual Review Template

Automated checks are useful, but human review is still important.

The table below creates a review template.

A reviewer can score each answer on:

```text
relevance
groundedness
citation quality
business usefulness
```

Score scale:

```text
1 = poor
2 = weak
3 = acceptable
4 = good
5 = excellent
```

Create manual review template

In [20]:
# Create manual review template from generated answers
manual_review_df = demo_answers_df[
    [
        "question",
        "ticker",
        "section_name",
        "answer",
        "citations"
    ]
].copy()

# Add blank review columns
manual_review_df["relevance_score_1_to_5"] = ""
manual_review_df["groundedness_score_1_to_5"] = ""
manual_review_df["citation_quality_score_1_to_5"] = ""
manual_review_df["business_usefulness_score_1_to_5"] = ""
manual_review_df["review_notes"] = ""

# Display manual review template
manual_review_df.head()

,question,ticker,section_name,answer,citations,relevance_score_1_to_5,groundedness_score_1_to_5,citation_quality_score_1_to_5,business_usefulness_score_1_to_5,review_notes
0,What AI and competition risks does NVIDIA ment...,NVDA,item_1a_risk_factors,"Based on the provided SEC evidence, NVIDIA men...","['NVDA 2026-02-25 10-K, item_1a_risk_factors, ...",,,,,
1,What cybersecurity risks does Microsoft mention?,MSFT,item_1a_risk_factors,Microsoft mentions the following cybersecurity...,"['MSFT 2025-07-30 10-K, item_1a_risk_factors, ...",,,,,
2,What supply chain risks does Tesla mention?,TSLA,item_1a_risk_factors,"Based on the provided SEC evidence, Tesla ment...","['TSLA 2026-01-29 10-K, item_1a_risk_factors, ...",,,,,
3,What competition risks does Apple describe?,AAPL,item_1a_risk_factors,"Based on the provided SEC evidence, Apple desc...","['AAPL 2025-10-31 10-K, item_1a_risk_factors, ...",,,,,
4,What semiconductor competition risks does AMD ...,AMD,item_1a_risk_factors,"Based on the provided SEC evidence, AMD mentio...","['AMD 2026-02-04 10-K, item_1a_risk_factors, c...",,,,,


Save evaluation outputs

In [21]:
# Set output paths
citation_eval_file = PROCESSED_DIR / "sec_10k_citation_evaluation.csv"
groundedness_eval_file = PROCESSED_DIR / "sec_10k_groundedness_proxy_eval.csv"
quality_flags_file = PROCESSED_DIR / "sec_10k_rag_quality_flags.csv"
evaluation_summary_file = PROCESSED_DIR / "sec_10k_rag_evaluation_summary.csv"
failure_cases_file = PROCESSED_DIR / "sec_10k_rag_failure_cases.csv"
manual_review_file = PROCESSED_DIR / "sec_10k_manual_review_template.csv"

# Save evaluation outputs
citation_eval_df.to_csv(citation_eval_file, index=False)
groundedness_eval_df.to_csv(groundedness_eval_file, index=False)
quality_flags_df.to_csv(quality_flags_file, index=False)
evaluation_summary.to_csv(evaluation_summary_file, index=False)
failure_cases_df.to_csv(failure_cases_file, index=False)
manual_review_df.to_csv(manual_review_file, index=False)

# Confirm saved files
print("Saved citation evaluation to:", citation_eval_file)
print("Saved groundedness proxy evaluation to:", groundedness_eval_file)
print("Saved quality flags to:", quality_flags_file)
print("Saved evaluation summary to:", evaluation_summary_file)
print("Saved failure cases to:", failure_cases_file)
print("Saved manual review template to:", manual_review_file)

Saved citation evaluation to: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\processed\sec_10k_citation_evaluation.csv
Saved groundedness proxy evaluation to: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\processed\sec_10k_groundedness_proxy_eval.csv
Saved quality flags to: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\processed\sec_10k_rag_quality_flags.csv
Saved evaluation summary to: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\processed\sec_10k_rag_evaluation_summary.csv
Saved failure cases to: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\processed\sec_10k_rag_failure_cases.csv
Saved manual review template to: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\processed\sec_10k_manual_review_template.csv


Create markdown evaluation report

In [22]:
# Set report path
evaluation_report_file = REPORTS_DIR / "rag_evaluation_report.md"

# Create report sections
report_sections = []

# Add title
report_sections.append("# RiskRadar AI: RAG Evaluation Report\n")

# Add overview
report_sections.append(
    "This report summarizes the first evaluation layer for RiskRadar AI.\n"
)

# Add summary metrics
report_sections.append("## Evaluation Summary\n")

for _, row in evaluation_summary.iterrows():
    report_sections.append(f"- **{row['metric']}**: {row['value']}")

# Add retrieval summary
report_sections.append("\n## Retrieval Summary\n")

if not retrieval_eval_summary.empty:
    for _, row in retrieval_eval_summary.iterrows():
        report_sections.append(f"- {row.to_dict()}")

# Add failure cases
report_sections.append("\n## Known Failure Cases\n")

for _, row in failure_cases_df.iterrows():
    report_sections.append(
        f"- **{row['failure_case']}**: {row['mitigation']}"
    )

# Add conclusion
report_sections.append(
    "\n## Conclusion\n"
    "The RAG system successfully generates cited answers from retrieved SEC filing evidence. "
    "The evaluation layer checks citation usage, citation validity, answer-evidence overlap, "
    "retrieval quality, and known failure cases. Future improvements should add stronger "
    "faithfulness scoring, query routing, and human review."
)

# Join report text
report_text = "\n".join(report_sections)

# Save report
evaluation_report_file.write_text(report_text, encoding="utf-8")

# Confirm report saved
print("Saved evaluation report to:", evaluation_report_file)

Saved evaluation report to: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\reports\rag_evaluation_report.md


Final checkpoint

In [23]:
# Create final checkpoint table
evaluation_checkpoint = pd.DataFrame({
    "output": [
        "Citation evaluation",
        "Groundedness proxy evaluation",
        "Quality flags",
        "Evaluation summary",
        "Failure cases",
        "Manual review template",
        "Markdown evaluation report"
    ],
    "path": [
        str(citation_eval_file),
        str(groundedness_eval_file),
        str(quality_flags_file),
        str(evaluation_summary_file),
        str(failure_cases_file),
        str(manual_review_file),
        str(evaluation_report_file)
    ],
    "exists": [
        citation_eval_file.exists(),
        groundedness_eval_file.exists(),
        quality_flags_file.exists(),
        evaluation_summary_file.exists(),
        failure_cases_file.exists(),
        manual_review_file.exists(),
        evaluation_report_file.exists()
    ]
})

# Display checkpoint table
evaluation_checkpoint

,output,path,exists
0,Citation evaluation,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...,True
1,Groundedness proxy evaluation,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...,True
2,Quality flags,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...,True
3,Evaluation summary,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...,True
4,Failure cases,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...,True
5,Manual review template,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...,True
6,Markdown evaluation report,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...,True


Final validation

In [24]:
# Validate notebook 09 outputs

# Check that all output files exist
all_outputs_exist = evaluation_checkpoint["exists"].all()

# Print output status
print("All evaluation outputs exist:", all_outputs_exist)

# Print key metrics
print("Demo answers evaluated:", len(citation_eval_df))
print("Source marker rate:", citation_eval_df["has_source_marker"].mean())
print("Average citations per answer:", citation_eval_df["num_citations"].mean())
print("Average citation validity rate:", citation_eval_df["citation_validity_rate"].mean())
print("Average answer-evidence overlap:", groundedness_eval_df["answer_evidence_overlap_rate"].mean())

# Stop if any output is missing
if not all_outputs_exist:
    raise ValueError("Some evaluation outputs are missing.")

# Confirm notebook completed successfully
print("Notebook 09 completed successfully.")

All evaluation outputs exist: True
Demo answers evaluated: 5
Source marker rate: 1.0
Average citations per answer: 3.0
Average citation validity rate: 1.0
Average answer-evidence overlap: 0.7694996619337391
Notebook 09 completed successfully.


## RAG Evaluation Conclusion

This notebook created the first evaluation layer for RiskRadar AI.

The project now has:

```text
generated RAG answers
→ citation validation
→ source marker checks
→ citation-to-chunk matching
→ answer-evidence overlap scoring
→ retrieval evaluation summary
→ failure case documentation
→ manual review template
→ evaluation report
```

Key lesson:

```text
A strong RAG project should not only generate answers.
It should also evaluate whether answers are grounded, cited, and useful.
```

The next notebook will add structured financial data.

```text
SEC XBRL company facts
→ financial metrics
→ ratios
→ combine structured data with RAG answers
```